# Clustering audio à partir des fichiers chroma

On utilise ici les features de Chordino qui permettent d'étudier les accords.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import seaborn as sns
import plotly.express as px
from plotly.offline import plot
import plotly.io as pio
from sklearn.manifold import MDS
import librosa
from scipy.cluster.hierarchy import linkage, dendrogram

In [2]:
dico_res = {"maj":[0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0],
"min":[0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0],
"dim_dim7":[0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,1,0,0],
"dim_min7":[0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,1,0],
"maj_min7":[0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,1,0],
"maj_maj7":[0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,1],
"min_min7":[0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,1,0,0,1,0],
"dim":[0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0],
"aug":[0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0]}

In [3]:
dico_notes = {
    "A" : 0,
    "A#" : 1,
    "Bb" : 1,
    "B" : 2,
    "C" : 3,
    "C#" : 4,
    "Db" : 4,
    "D" : 5,
    "D#" : 6,
    "Eb" : 6,
    "E" : 7,
    "F" : 8,
    "F#" : 9,
    "Gb" : 9,
    "G" : 10,
    "G#" : 11,
    "Ab" : 11,
    "N" : -1
}

## Récupération d'un fichier et transformation en features

In [ ]:
def get_matrix(chunk):
    li = []
    for accord in chunk:
        partition = accord.split("_")
        note = partition[0]
        if dico_notes[note] > 0:
            liste = [0]*12
            liste[dico_notes[note]]=1
            acc = "_".join(partition[1:])
            liste += dico_res[acc]
            li.append(liste)
    return np.array(li)

In [28]:
def get_chunks(filename):
    data = pd.read_csv(filename, sep=",", header=None)
    chunks = {}
    chunk = []
    title = None
    for i, row in data.iterrows():
        if pd.notna(row[0]):
            if title is not None:
                chunks[title] = chunk
            title = row[0]
            chunk = []
        else:
            chunk.append(row[2])
    
    if title is not None:
        chunks[title] = chunk
    return chunks

def dico_chunks(filename):
    chunks = get_chunks(filename)
    return {k: get_matrix(v) for k, v in chunks.items() if len(v) > 0}

## Clustering des chunks